# SafeFundus — Notebook 10B: RFMiD 2.0 External Validation — FIXED

This is the corrected external-validation notebook.

## Fixes in this version

RFMiD 2.0 contains mixed-case image extensions:

- `.jpg`
- `.JPG`

Therefore this notebook **does not construct filenames by assuming `.jpg`**.  
It inventories the real Test_set files and maps images to CSV rows using the numeric image ID.

The uploaded RFMiD 2.0 archive also contains a small exact-ID alignment issue:

- Test CSV label IDs without an exact image-ID match: `151, 191, 405, 490`
- Test image IDs without an exact CSV label row: `251, 291, 404, 489`

Those unmatched pairs are **reported and excluded** from external inference.  
No manual mapping is guessed.

The frozen SafeFundus pipeline remains unchanged:

\[
\text{Swin-Tiny + ASL + NORMAL}
\rightarrow
\text{Temperature Scaling}
\rightarrow
\text{Normal-Aware JointEnergy}
+
\text{20-pass MC Dropout}
\rightarrow
R(x)=0.5J_N+0.5U_N
\rightarrow
\text{frozen referral threshold}
\]

Nothing is trained or tuned on RFMiD 2.0.


In [1]:
from pathlib import Path
import json, random, shutil, zipfile, warnings, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    roc_auc_score, average_precision_score,
    confusion_matrix, roc_curve,
)

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torchvision.models import Swin_T_Weights
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")

NOTEBOOK_STEM = "10B_RFMiD2_External_Validation_Fixed"
OUTPUT_DIR = WORK_ROOT / NOTEBOOK_STEM
TABLE_DIR = OUTPUT_DIR / "tables"
FIG_DIR = OUTPUT_DIR / "figures"
SEED_DIR = OUTPUT_DIR / "per_seed"
for p in [OUTPUT_DIR, TABLE_DIR, FIG_DIR, SEED_DIR]:
    p.mkdir(parents=True, exist_ok=True)

SEEDS = [42, 123, 2026]
MC_PASSES = 20
KNOWN = ["DR","MH","ODC","TSLN","DN","MYA","ARMD","BRVO","ODP","ODE","LS","RS","CSR","CRS"]
INTERMEDIATE = ["CRVO","RPEC"]
MODEL_CLASSES = ["NORMAL"] + KNOWN
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("DEVICE:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


DEVICE: cuda
GPU: Tesla T4


## Attach these four Kaggle inputs

1. RFMiD 2.0 dataset
2. `06C_SwinTiny_ASL_NormalAware_MultiSeed_Classifier`
3. `08_MC_Dropout_Uncertainty`
4. `09_SafeFundus_Fusion_Calibration_Ablation`

The code searches `/kaggle/input` automatically.


In [2]:
# -----------------------------
# Locate RFMiD 2.0
# -----------------------------
test_csv_candidates = list(INPUT_ROOT.rglob("RFMiD_2_Testing_labels.csv"))
if not test_csv_candidates:
    test_csv_candidates = [
        p for p in INPUT_ROOT.rglob("*.csv")
        if p.name.lower() == "rfmid_2_testing_labels.csv"
    ]
if not test_csv_candidates:
    raise FileNotFoundError("Attach the RFMiD 2.0 Kaggle dataset.")

TEST_CSV = test_csv_candidates[0]
TEST_DIR = TEST_CSV.parent
RFMID2_ROOT = TEST_DIR.parent
TRAIN_DIR = RFMID2_ROOT / "Training_set"
VAL_DIR = RFMID2_ROOT / "Validation_set"
TRAIN_CSV = TRAIN_DIR / "RFMiD_2_Training_labels.csv"
VAL_CSV = VAL_DIR / "RFMiD_2_Validation_labels.csv"

print("RFMiD 2.0 root:", RFMID2_ROOT)
print("Train CSV:", TRAIN_CSV)
print("Validation CSV:", VAL_CSV)
print("Test CSV:", TEST_CSV)

def read_rfmid2(path):
    df = None
    used = None
    for enc in ["utf-8-sig", "cp1252", "latin1"]:
        try:
            df = pd.read_csv(path, encoding=enc)
            used = enc
            break
        except UnicodeDecodeError:
            pass
    if df is None:
        raise RuntimeError(f"Could not decode {path}")

    df.columns = [str(c).replace("\xa0"," ").strip() for c in df.columns]
    unnamed = [c for c in df.columns if c.lower().startswith("unnamed")]
    audit = {}
    for c in unnamed:
        vals = df[c].dropna()
        audit[c] = {
            "non_null_count": int(len(vals)),
            "unique_values": [float(v) for v in sorted(vals.unique().tolist())],
        }
    if unnamed:
        df = df.drop(columns=unnamed)

    for c in df.columns:
        if c != "ID":
            df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)
    return df, {"encoding":used, "dropped_unnamed":audit}

train_df, train_csv_audit = read_rfmid2(TRAIN_CSV)
val_df, val_csv_audit = read_rfmid2(VAL_CSV)
test_df, test_csv_audit = read_rfmid2(TEST_CSV)

print("Rows:", len(train_df), len(val_df), len(test_df),
      "Total:", len(train_df)+len(val_df)+len(test_df))
print("Test encoding:", test_csv_audit["encoding"])
print("Dropped unnamed test field:", test_csv_audit["dropped_unnamed"])


RFMiD 2.0 root: /kaggle/input/datasets/shouravdebnath/rfmid-2-0/RFMiD2_0
Train CSV: /kaggle/input/datasets/shouravdebnath/rfmid-2-0/RFMiD2_0/Training_set/RFMiD_2_Training_labels.csv
Validation CSV: /kaggle/input/datasets/shouravdebnath/rfmid-2-0/RFMiD2_0/Validation_set/RFMiD_2_Validation_labels.csv
Test CSV: /kaggle/input/datasets/shouravdebnath/rfmid-2-0/RFMiD2_0/Test_set/RFMiD_2_Testing_labels.csv
Rows: 509 177 174 Total: 860
Test encoding: cp1252
Dropped unnamed test field: {'Unnamed: 52': {'non_null_count': 9, 'unique_values': [1.0, 2.0]}}


In [3]:
# -----------------------------
# Label harmonization + robust image/CSV alignment
# -----------------------------

for c in ["ID", "WNL"] + KNOWN + INTERMEDIATE:
    if c not in test_df.columns:
        raise ValueError(
            f"Missing required RFMiD 2.0 column: {c}"
        )

PATHOLOGY = [
    c for c in test_df.columns
    if c not in {"ID", "WNL"}
]

UNKNOWN_EXT = [
    c for c in PATHOLOGY
    if c not in KNOWN + INTERMEDIATE
]


def group_row(row):
    positive = {
        c for c in PATHOLOGY
        if float(row[c]) > 0
    }

    if float(row["WNL"]) > 0 and not positive:
        return "Normal"

    if positive & set(INTERMEDIATE):
        return "Contains_Intermediate"

    has_known = bool(
        positive & set(KNOWN)
    )

    has_unknown = bool(
        positive & set(UNKNOWN_EXT)
    )

    if has_known and has_unknown:
        return "Mixed_Known_Unknown"

    if has_known:
        return "Known_Only"

    if has_unknown:
        return "Unknown_Only"

    return "Other"


def audit_split(df, name):
    out = df.copy()

    out["External_Group"] = out.apply(
        group_row,
        axis=1,
    )

    any_pathology = (
        out[PATHOLOGY]
        .gt(0)
        .any(axis=1)
    )

    wnl = (
        out["WNL"]
        .gt(0)
    )

    mismatch = (
        (
            wnl
            & any_pathology
        )
        |
        (
            (~wnl)
            & (~any_pathology)
        )
    )

    audit = {
        "Split": name,
        "CSV_Rows": int(len(out)),
        "WNL_Mismatch_Count": int(mismatch.sum()),
    }

    audit.update({
        f"Group_{k}": int(v)
        for k, v in (
            out["External_Group"]
            .value_counts()
            .items()
        )
    })

    return out, audit


train_df, train_group_audit = audit_split(
    train_df,
    "Training_set",
)

val_df, val_group_audit = audit_split(
    val_df,
    "Validation_set",
)

test_df, test_group_audit = audit_split(
    test_df,
    "Test_set",
)


# ============================================================
# IMPORTANT FIX:
# Inventory the real files and use their actual extension/case.
# ============================================================

VALID_IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
}


def norm_id(x):
    return str(
        int(
            float(x)
        )
    )


def build_image_index(folder):
    image_index = {}
    extension_counts = {}

    for p in folder.iterdir():

        if not p.is_file():
            continue

        if (
            p.suffix.lower()
            not in VALID_IMAGE_EXTENSIONS
        ):
            continue

        image_id = norm_id(
            p.stem
        )

        if image_id in image_index:
            raise ValueError(
                f"Duplicate image ID {image_id} "
                f"inside {folder}"
            )

        # Store the ACTUAL filename, including .jpg / .JPG case.
        image_index[
            image_id
        ] = str(p)

        extension_counts[
            p.suffix
        ] = (
            extension_counts.get(
                p.suffix,
                0,
            )
            + 1
        )

    return (
        image_index,
        extension_counts,
    )


TRAIN_IMAGE_INDEX, TRAIN_EXTENSION_COUNTS = (
    build_image_index(
        TRAIN_DIR
    )
)

VAL_IMAGE_INDEX, VAL_EXTENSION_COUNTS = (
    build_image_index(
        VAL_DIR
    )
)

TEST_IMAGE_INDEX, TEST_EXTENSION_COUNTS = (
    build_image_index(
        TEST_DIR
    )
)


def add_exact_image_alignment(
    df,
    image_index,
):
    out = df.copy()

    out["_Normalized_ID"] = (
        out["ID"]
        .apply(
            norm_id
        )
    )

    out["image_path"] = (
        out["_Normalized_ID"]
        .map(
            image_index
        )
    )

    out["Has_Exact_Image"] = (
        out["image_path"]
        .notna()
    )

    return out


train_df = add_exact_image_alignment(
    train_df,
    TRAIN_IMAGE_INDEX,
)

val_df = add_exact_image_alignment(
    val_df,
    VAL_IMAGE_INDEX,
)

test_df = add_exact_image_alignment(
    test_df,
    TEST_IMAGE_INDEX,
)


def alignment_audit(
    df,
    image_index,
    split_name,
    extension_counts,
):
    label_ids = set(
        df["_Normalized_ID"]
        .tolist()
    )

    image_ids = set(
        image_index.keys()
    )

    labels_without_image = sorted(
        label_ids - image_ids,
        key=int,
    )

    images_without_label = sorted(
        image_ids - label_ids,
        key=int,
    )

    return {
        "Split": split_name,
        "CSV_Rows": int(len(df)),
        "Image_Files": int(len(image_index)),
        "Exact_ID_Matches": int(
            df["Has_Exact_Image"]
            .sum()
        ),
        "Label_IDs_Without_Image": ",".join(
            labels_without_image
        ),
        "Images_Without_Label_ID": ",".join(
            images_without_label
        ),
        "Extension_Counts": json.dumps(
            extension_counts,
            sort_keys=True,
        ),
    }


alignment_df = pd.DataFrame([
    alignment_audit(
        train_df,
        TRAIN_IMAGE_INDEX,
        "Training_set",
        TRAIN_EXTENSION_COUNTS,
    ),
    alignment_audit(
        val_df,
        VAL_IMAGE_INDEX,
        "Validation_set",
        VAL_EXTENSION_COUNTS,
    ),
    alignment_audit(
        test_df,
        TEST_IMAGE_INDEX,
        "Test_set",
        TEST_EXTENSION_COUNTS,
    ),
])

print("\nIMAGE / CSV ALIGNMENT AUDIT")
display(
    alignment_df
)

alignment_df.to_csv(
    TABLE_DIR /
    "rfmid2_image_label_alignment_audit.csv",
    index=False,
)


group_audit_df = pd.DataFrame([
    train_group_audit,
    val_group_audit,
    test_group_audit,
])

print("\nLABEL / GROUP AUDIT")
display(
    group_audit_df
)

group_audit_df.to_csv(
    TABLE_DIR /
    "rfmid2_dataset_audit.csv",
    index=False,
)


print(
    "\nExternal unknown label columns:",
    len(UNKNOWN_EXT),
)

print(
    UNKNOWN_EXT
)

print(
    "\nRaw Test_set groups before exact image matching:"
)

display(
    test_df["External_Group"]
    .value_counts()
    .rename_axis("Group")
    .reset_index(name="Count")
)


# ------------------------------------------------------------
# Use ONLY exact CSV-ID <-> image-ID pairs.
# No guessed relabeling.
# ------------------------------------------------------------

test_unmatched_labels = (
    test_df[
        ~test_df["Has_Exact_Image"]
    ]
    .copy()
)

if len(test_unmatched_labels) > 0:
    print(
        "\nWARNING — excluding Test_set label rows "
        "without an exact image-ID match:"
    )

    display(
        test_unmatched_labels[
            [
                "ID",
                "External_Group",
            ]
        ]
    )


test_evaluable_df = (
    test_df[
        test_df["Has_Exact_Image"]
    ]
    .copy()
)

print(
    "\nEvaluable external Test_set:",
    len(test_evaluable_df),
    "/",
    len(test_df),
)

print(
    "\nEvaluable Test_set groups:"
)

display(
    test_evaluable_df[
        "External_Group"
    ]
    .value_counts()
    .rename_axis("Group")
    .reset_index(name="Count")
)


familiar_df = (
    test_evaluable_df[
        test_evaluable_df[
            "External_Group"
        ]
        .isin(
            [
                "Normal",
                "Known_Only",
            ]
        )
    ]
    .copy()
)

unknown_df = (
    test_evaluable_df[
        test_evaluable_df[
            "External_Group"
        ]
        == "Unknown_Only"
    ]
    .copy()
)

mixed_df = (
    test_evaluable_df[
        test_evaluable_df[
            "External_Group"
        ]
        == "Mixed_Known_Unknown"
    ]
    .copy()
)

inter_df = (
    test_evaluable_df[
        test_evaluable_df[
            "External_Group"
        ]
        == "Contains_Intermediate"
    ]
    .copy()
)

print(
    "\nFINAL EVALUABLE COUNTS"
)

print(
    "Familiar:",
    len(familiar_df),
)

print(
    "Unknown-only:",
    len(unknown_df),
)

print(
    "Mixed:",
    len(mixed_df),
)

print(
    "Intermediate:",
    len(inter_df),
)

# Based on the uploaded RFMiD 2.0 archive, these should be:
# Familiar = 98 (52 normal + 46 known-only)
# Unknown-only = 50
# Mixed = 19
# Intermediate = 3



IMAGE / CSV ALIGNMENT AUDIT


,Split,CSV_Rows,Image_Files,Exact_ID_Matches,Label_IDs_Without_Image,Images_Without_Label_ID,Extension_Counts
0,Training_set,509,509,507,"89,216","79,222","{"".JPG"": 52, "".jpg"": 457}"
1,Validation_set,177,177,177,,,"{"".JPG"": 21, "".jpg"": 156}"
2,Test_set,174,174,170,"151,191,405,490","251,291,404,489","{"".JPG"": 21, "".jpg"": 153}"



LABEL / GROUP AUDIT


,Split,CSV_Rows,WNL_Mismatch_Count,Group_Normal,Group_Known_Only,Group_Unknown_Only,Group_Mixed_Known_Unknown,Group_Contains_Intermediate
0,Training_set,509,0,157,153,146,44,9
1,Validation_set,177,0,53,58,50,12,4
2,Test_set,174,0,52,46,54,19,3



External unknown label columns: 34
['AH', 'AION', 'CB', 'CF', 'CL', 'CME', 'CNV', 'CRAO', 'CWS', 'CSC', 'EDN', 'ERM', 'GRT', 'HPED', 'HR', 'MCA', 'ME', 'MHL', 'MS', 'ON', 'OPDM', 'PRH', 'RD', 'RHL', 'RTR', 'RP', 'RT', 'SOFE', 'ST', 'TD', 'TV', 'VS', 'HTN', 'IIH']

Raw Test_set groups before exact image matching:


,Group,Count
0,Unknown_Only,54
1,Normal,52
2,Known_Only,46
3,Mixed_Known_Unknown,19
4,Contains_Intermediate,3



WARNING — excluding Test_set label rows without an exact image-ID match:


,ID,External_Group
0,151,Unknown_Only
3,191,Unknown_Only
37,405,Unknown_Only
60,490,Unknown_Only



Evaluable external Test_set: 170 / 174

Evaluable Test_set groups:


,Group,Count
0,Normal,52
1,Unknown_Only,50
2,Known_Only,46
3,Mixed_Known_Unknown,19
4,Contains_Intermediate,3



FINAL EVALUABLE COUNTS
Familiar: 98
Unknown-only: 50
Mixed: 19
Intermediate: 3


In [4]:
# -----------------------------
# Locate frozen SafeFundus outputs
# -----------------------------
def locate_or_extract(source_name, signature, extract_name):
    zips = list(INPUT_ROOT.rglob(f"{source_name}.zip"))
    if zips:
        dst = WORK_ROOT / extract_name
        if dst.exists():
            shutil.rmtree(dst)
        dst.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zips[0]) as zf:
            zf.extractall(dst)
        print("Found:", zips[0])
        return dst

    sigs = list(INPUT_ROOT.rglob(signature))
    for p in sigs:
        if "seed_42" not in str(p):
            continue
        if "per_seed" in p.parts:
            idx = p.parts.index("per_seed")
            return Path(*p.parts[:idx])
        for parent in p.parents:
            if parent.name == "seed_42":
                return parent.parent
    raise FileNotFoundError(f"Attach {source_name}")

ROOT06 = locate_or_extract(
    "06C_SwinTiny_ASL_NormalAware_MultiSeed_Classifier",
    "swin_tiny_asl_normalaware_seed_42.pth",
    "06c_rfmid2_external"
)
ROOT08 = locate_or_extract(
    "08_MC_Dropout_Uncertainty",
    "mc_dropout_seed_summary.json",
    "08_rfmid2_external"
)
ROOT09 = locate_or_extract(
    "09_SafeFundus_Fusion_Calibration_Ablation",
    "fusion_validation_parameters.json",
    "09_rfmid2_external"
)

def frozen_seed(seed):
    ckpt = ROOT06/"checkpoints"/f"swin_tiny_asl_normalaware_seed_{seed}.pth"

    mc_json = ROOT08/"per_seed"/f"seed_{seed}"/"mc_dropout_seed_summary.json"
    if not mc_json.exists():
        mc_json = ROOT08/f"seed_{seed}"/"mc_dropout_seed_summary.json"

    fjson = ROOT09/"per_seed"/f"seed_{seed}"/"fusion_validation_parameters.json"
    if not fjson.exists():
        fjson = ROOT09/f"seed_{seed}"/"fusion_validation_parameters.json"

    for p in [ckpt, mc_json, fjson]:
        if not p.exists():
            raise FileNotFoundError(str(p))

    with open(mc_json) as f:
        mc = json.load(f)
    with open(fjson) as f:
        fp = json.load(f)

    return {
        "checkpoint":ckpt,
        "mc_mu":float(mc["validation_uncertainty_mean"]),
        "mc_sd":float(mc["validation_uncertainty_sd"]),
        "joint_branch":fp["jointenergy_branch_parameters"],
        "joint_mu":float(fp["jointenergy_final_validation_mean"]),
        "joint_sd":float(fp["jointenergy_final_validation_sd"]),
        "alpha_j":float(fp["fusion_alpha_jointenergy"]),
        "alpha_u":float(fp["fusion_alpha_mc_dropout"]),
        "fusion_threshold":float(fp["fusion_primary_threshold"]),
        "temperature":float(fp["temperature"]),
        "thresholds":{k:float(v) for k,v in fp["calibrated_thresholds"].items()},
    }

for s in SEEDS:
    f = frozen_seed(s)
    print(s, "T=", round(f["temperature"],4),
          "fusion threshold=", round(f["fusion_threshold"],4))


Found: /kaggle/input/notebooks/shouravdebnath/06c-swintiny-asl-normalaware-multiseed-classifier/06C_SwinTiny_ASL_NormalAware_MultiSeed_Classifier.zip
Found: /kaggle/input/notebooks/shouravdebnath/08-mc-dropout-uncertainty/08_MC_Dropout_Uncertainty.zip
Found: /kaggle/input/notebooks/shouravdebnath/09-safefundus-fusion-calibration-ablation/09_SafeFundus_Fusion_Calibration_Ablation.zip
42 T= 0.4221 fusion threshold= 0.8621
123 T= 0.6491 fusion threshold= 0.956
2026 T= 0.6272 fusion threshold= 0.8439


In [5]:
# -----------------------------
# Dataset / model / transforms
# -----------------------------

weights = (
    Swin_T_Weights
    .IMAGENET1K_V1
)

mean = (
    weights.transforms().mean
)

std = (
    weights.transforms().std
)

eval_tf = transforms.Compose([
    transforms.Resize(
        (
            224,
            224,
        )
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=mean,
        std=std,
    ),
])


# Paths were resolved from the ACTUAL Test_set file inventory.
# This verifies them but never changes their extension/case.
for split_name, df in [
    ("familiar", familiar_df),
    ("unknown", unknown_df),
    ("mixed", mixed_df),
    ("intermediate", inter_df),
]:

    missing = [
        str(p)
        for p in df[
            "image_path"
        ]
        if (
            pd.isna(p)
            or not Path(
                p
            ).exists()
        )
    ]

    if missing:
        raise FileNotFoundError(
            f"{split_name}: "
            f"{len(missing)} unresolved image paths. "
            f"Example: {missing[:3]}"
        )

print(
    "✓ All evaluable RFMiD 2.0 image paths verified."
)

print(
    "Test image extensions:",
    TEST_EXTENSION_COUNTS,
)


class ExtDataset(
    Dataset
):
    def __init__(
        self,
        df,
    ):
        self.df = (
            df
            .reset_index(
                drop=True
            )
        )

    def __len__(
        self
    ):
        return len(
            self.df
        )

    def __getitem__(
        self,
        i,
    ):
        r = (
            self.df
            .iloc[i]
        )

        im = Image.open(
            r[
                "image_path"
            ]
        ).convert(
            "RGB"
        )

        im = eval_tf(
            im
        )

        target = [
            float(
                r[
                    "WNL"
                ]
            )
        ] + [
            float(
                r[
                    c
                ]
            )
            for c in KNOWN
        ]

        return {
            "image":
                im,

            "target":
                torch.tensor(
                    target,
                    dtype=
                        torch.float32,
                ),

            "id":
                norm_id(
                    r[
                        "ID"
                    ]
                ),

            "group":
                r[
                    "External_Group"
                ],
        }


def loader(
    df,
):
    return DataLoader(
        ExtDataset(
            df
        ),

        batch_size=
            32,

        shuffle=
            False,

        num_workers=
            2,

        pin_memory=
            True,

        persistent_workers=
            True,
    )


def build_model():
    m = models.swin_t(
        weights=None
    )

    n = (
        m.head.in_features
    )

    m.head = nn.Sequential(
        nn.Dropout(
            0.30
        ),
        nn.Linear(
            n,
            15,
        ),
    )

    return m


def load_model(
    path,
):
    m = (
        build_model()
    )

    ck = torch.load(
        path,
        map_location="cpu",
    )

    m.load_state_dict(
        ck[
            "model_state_dict"
        ]
    )

    return (
        m.to(
            DEVICE
        )
        .eval()
    )


def sigmoid_np(
    x,
):
    return (
        1
        /
        (
            1
            + np.exp(
                -np.clip(
                    x,
                    -60,
                    60,
                )
            )
        )
    )


✓ All evaluable RFMiD 2.0 image paths verified.
Test image extensions: {'.JPG': 21, '.jpg': 153}


In [6]:
# -----------------------------
# Inference functions
# -----------------------------
@torch.no_grad()
def deterministic(model, dl, T):
    ids, groups, ys, zs = [], [], [], []
    model.eval()
    for b in tqdm(dl, desc="Deterministic", leave=False):
        x = b["image"].to(DEVICE, non_blocking=True)
        z = model(x).float().cpu().numpy()
        zs.append(z); ys.append(b["target"].numpy())
        ids.extend(b["id"]); groups.extend(b["group"])
    z = np.concatenate(zs); y = np.concatenate(ys)
    return {
        "ids":ids, "groups":groups, "targets":y, "logits":z,
        "prob_uncal":sigmoid_np(z), "prob_cal":sigmoid_np(z/T)
    }

def enable_head_dropout(model):
    model.eval()
    found = 0
    for mod in model.head.modules():
        if isinstance(mod, nn.Dropout):
            mod.train(); found += 1
    if not found:
        raise RuntimeError("No prediction-head dropout found.")

@torch.no_grad()
def mc_dropout(model, dl, frozen):
    enable_head_dropout(model)
    ids, groups, raws = [], [], []
    for b in tqdm(dl, desc=f"MC x{MC_PASSES}", leave=False):
        x = b["image"].to(DEVICE, non_blocking=True)
        ps = []
        for _ in range(MC_PASSES):
            with torch.cuda.amp.autocast(enabled=DEVICE.type=="cuda"):
                p = torch.sigmoid(model(x))
            ps.append(p.float().cpu().numpy())
        stack = np.stack(ps, axis=0)
        raw = stack.var(axis=0, ddof=0).mean(axis=1)
        raws.append(raw)
        ids.extend(b["id"]); groups.extend(b["group"])
    raw = np.concatenate(raws)
    return {
        "ids":ids, "groups":groups, "raw":raw,
        "norm":(raw-frozen["mc_mu"])/frozen["mc_sd"]
    }

def jointenergy(logits, frozen):
    en = np.logaddexp(0.0, logits[:,0])
    ed = np.logaddexp(0.0, logits[:,1:]).sum(axis=1)
    p = frozen["joint_branch"]
    zn = (en-float(p["mu_normal"]))/float(p["sd_normal"])
    zd = (ed-float(p["mu_disease"]))/float(p["sd_disease"])
    raw = -np.maximum(zn,zd)
    norm = (raw-frozen["joint_mu"])/frozen["joint_sd"]
    return norm

def disease_pred(prob_cal, frozen):
    t = np.array([frozen["thresholds"][c] for c in KNOWN], dtype=float)
    return prob_cal[:,1:] >= t[None,:]


In [7]:
# -----------------------------
# Metrics
# -----------------------------
def fpr95(y, s):
    fpr,tpr,_ = roc_curve(y,s)
    ok = np.where(tpr>=0.95)[0]
    return float(np.min(fpr[ok])) if len(ok) else np.nan

def ood_metrics(fam, unk):
    y = np.r_[np.zeros(len(fam),dtype=int), np.ones(len(unk),dtype=int)]
    s = np.r_[fam,unk]
    return {
        "unknown_auroc":float(roc_auc_score(y,s)),
        "unknown_aupr":float(average_precision_score(y,s)),
        "fpr_at_95_tpr":fpr95(y,s),
    }

def ece(y,p,n_bins=15):
    y=y.reshape(-1); p=p.reshape(-1)
    edges=np.linspace(0,1,n_bins+1); out=0.0
    for i in range(n_bins):
        lo,hi=edges[i],edges[i+1]
        m=(p>=lo)&((p<=hi) if i==n_bins-1 else (p<hi))
        if m.any():
            out += float(m.mean())*abs(float(p[m].mean())-float(y[m].mean()))
    return float(out)

def fam_class_metrics(det, frozen):
    yt=det["targets"][:,1:].astype(int)
    yp=disease_pred(det["prob_cal"], frozen).astype(int)
    pr=det["prob_cal"][:,1:]
    pos=yt.sum(axis=0)>0
    aps=[]; aucs=[]
    for c in range(14):
        if yt[:,c].sum()>0:
            aps.append(average_precision_score(yt[:,c],pr[:,c]))
        if len(np.unique(yt[:,c]))==2:
            aucs.append(roc_auc_score(yt[:,c],pr[:,c]))
    return {
        "known_classes_with_positives":int(pos.sum()),
        "disease_macro_f1_all14":float(f1_score(yt,yp,average="macro",zero_division=0)),
        "disease_macro_f1_evaluable":float(f1_score(yt[:,pos],yp[:,pos],average="macro",zero_division=0)) if pos.any() else np.nan,
        "disease_micro_f1":float(f1_score(yt,yp,average="micro",zero_division=0)),
        "disease_mAP":float(np.mean(aps)) if aps else np.nan,
        "disease_macro_auroc":float(np.mean(aucs)) if aucs else np.nan,
    }

def per_class_metrics(det, frozen):
    yt=det["targets"][:,1:].astype(int)
    yp=disease_pred(det["prob_cal"], frozen).astype(int)
    pr=det["prob_cal"][:,1:]
    rows=[]
    for c,name in enumerate(KNOWN):
        tn,fp,fn,tp=confusion_matrix(yt[:,c],yp[:,c],labels=[0,1]).ravel()
        rows.append({
            "Disease":name,
            "External_Positive_Count":int(yt[:,c].sum()),
            "Threshold":frozen["thresholds"][name],
            "Precision":precision_score(yt[:,c],yp[:,c],zero_division=0),
            "Sensitivity_Recall":recall_score(yt[:,c],yp[:,c],zero_division=0),
            "Specificity":tn/(tn+fp) if (tn+fp)>0 else np.nan,
            "F1":f1_score(yt[:,c],yp[:,c],zero_division=0),
            "AUROC":roc_auc_score(yt[:,c],pr[:,c]) if len(np.unique(yt[:,c]))==2 else np.nan,
            "Average_Precision":average_precision_score(yt[:,c],pr[:,c]) if yt[:,c].sum()>0 else np.nan,
            "TP":int(tp),"FP":int(fp),"TN":int(tn),"FN":int(fn),
        })
    return pd.DataFrame(rows)


## Run the frozen model

The same three seeds are evaluated independently.

No RFMiD 2.0 statistic is used to change the model, normalization, fusion, temperature, or thresholds.


In [8]:
# -----------------------------
# Main external evaluation
# -----------------------------
summaries=[]
bootstrap=[]

for seed in SEEDS:
    print("\n"+"#"*72)
    print("RFMiD 2.0 — seed",seed)
    print("#"*72)

    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

    fz=frozen_seed(seed)
    model=load_model(fz["checkpoint"])
    dls={
        "familiar":loader(familiar_df),
        "unknown":loader(unknown_df),
        "mixed":loader(mixed_df),
        "intermediate":loader(inter_df),
    }

    det={}; mc={}; J={}; R={}; ref={}
    for split,dl in dls.items():
        print("\n",split)
        det[split]=deterministic(model,dl,fz["temperature"])
        mc[split]=mc_dropout(model,dl,fz)
        if det[split]["ids"] != mc[split]["ids"]:
            raise ValueError("Deterministic/MC ID order mismatch")
        J[split]=jointenergy(det[split]["logits"],fz)
        R[split]=fz["alpha_j"]*J[split] + fz["alpha_u"]*mc[split]["norm"]
        ref[split]=R[split] > fz["fusion_threshold"]

    fam_groups=np.array(det["familiar"]["groups"])
    normal_mask=fam_groups=="Normal"
    known_mask=fam_groups=="Known_Only"

    unk_pred=disease_pred(det["unknown"]["prob_cal"],fz)
    unk_any=unk_pred.any(axis=1)
    unk_no=~unk_any
    explicit_normal=det["unknown"]["prob_cal"][:,0] >= fz["thresholds"]["NORMAL"]
    residual=~ref["unknown"]
    rare_normal=residual & unk_no
    strict_false=rare_normal & explicit_normal
    forced_known=residual & unk_any

    cls=fam_class_metrics(det["familiar"],fz)
    ood=ood_metrics(R["familiar"],R["unknown"])

    yt=det["familiar"]["targets"]
    uncal=det["familiar"]["prob_uncal"]
    cal=det["familiar"]["prob_cal"]

    summary={
        "seed":seed,
        "external_test_csv_rows":len(test_df),
        "external_test_evaluable_images":len(test_evaluable_df),
        "external_test_excluded_unmatched_label_rows":len(test_unmatched_labels),
        "external_familiar_images":len(familiar_df),
        "external_unknown_only_images":len(unknown_df),
        "external_mixed_images":len(mixed_df),
        "familiar_false_referral_rate":float(ref["familiar"].mean()),
        "normal_false_referral_rate":float(ref["familiar"][normal_mask].mean()),
        "known_only_false_referral_rate":float(ref["familiar"][known_mask].mean()),
        "unknown_referral_sensitivity":float(ref["unknown"].mean()),
        "mixed_referral_rate":float(ref["mixed"].mean()) if len(mixed_df) else np.nan,
        "rare_to_normal_error_rate":float(rare_normal.mean()),
        "explicit_normal_false_reassurance_rate":float(strict_false.mean()),
        "unknown_forced_into_known_rate":float(forced_known.mean()),
        **ood, **cls,
        "external_ece_before":ece(yt,uncal),
        "external_ece_after":ece(yt,cal),
        "external_brier_before":float(np.mean((yt-uncal)**2)),
        "external_brier_after":float(np.mean((yt-cal)**2)),
    }
    summaries.append(summary)

    sd=SEED_DIR/f"seed_{seed}"
    sd.mkdir(parents=True,exist_ok=True)
    per_class_metrics(det["familiar"],fz).to_csv(
        sd/"external_per_class_disease_metrics.csv",index=False
    )

    for split in ["familiar","unknown","mixed","intermediate"]:
        d=det[split]
        ep=disease_pred(d["prob_cal"],fz)
        x=pd.DataFrame({
            "ID":d["ids"],
            "External_Group":d["groups"],
            "JointEnergy_Normalized":J[split],
            "MC_Dropout_Normalized":mc[split]["norm"],
            "Fusion_R":R[split],
            "Referral":ref[split].astype(int),
            "Num_Known_Diseases_Predicted":ep.sum(axis=1).astype(int),
            "Final_Normal_if_Not_Referred":((~ep.any(axis=1))&(~ref[split])).astype(int),
        })
        for c,name in enumerate(MODEL_CLASSES):
            x[f"true_{name}"]=d["targets"][:,c].astype(int)
            x[f"prob_cal_{name}"]=d["prob_cal"][:,c]
        if split=="unknown":
            x["Rare_to_Normal_Error"]=rare_normal.astype(int)
            x["Explicit_Normal_False_Reassurance"]=strict_false.astype(int)
            x["Forced_Into_Known"]=forced_known.astype(int)
        x.to_csv(sd/f"rfmid2_{split}_predictions.csv",index=False)

    with open(sd/"external_seed_summary.json","w") as f:
        json.dump(summary,f,indent=2)

    # Bootstrap 95% CI for primary rates.
    rng=np.random.default_rng(seed+10000)
    arrays={
        "familiar_false_referral_rate":ref["familiar"].astype(float),
        "unknown_referral_sensitivity":ref["unknown"].astype(float),
        "rare_to_normal_error_rate":rare_normal.astype(float),
    }
    for metric,vals in arrays.items():
        sims=[]
        n=len(vals)
        for _ in range(2000):
            idx=rng.integers(0,n,size=n)
            sims.append(float(vals[idx].mean()))
        bootstrap.append({
            "seed":seed,"Metric":metric,"Estimate":float(vals.mean()),
            "CI95_Lower":float(np.quantile(sims,.025)),
            "CI95_Upper":float(np.quantile(sims,.975)),
        })

    plt.figure(figsize=(8,5))
    plt.hist(R["familiar"],bins=20,density=True,alpha=.6,label="External familiar")
    plt.hist(R["unknown"],bins=15,density=True,alpha=.6,label="External unknown-only")
    plt.axvline(fz["fusion_threshold"],linestyle="--",label="Frozen RFMiD threshold")
    plt.xlabel("SafeFundus Fusion Score R(x)")
    plt.ylabel("Density")
    plt.title(f"RFMiD 2.0 External Scores — Seed {seed}")
    plt.legend(); plt.tight_layout()
    plt.savefig(FIG_DIR/f"rfmid2_external_fusion_seed_{seed}.png",dpi=200,bbox_inches="tight")
    plt.close()

    print(json.dumps(summary,indent=2))
    del model,dls,det,mc
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()



########################################################################
RFMiD 2.0 — seed 42
########################################################################

 familiar


Deterministic:   0%|          | 0/4 [00:00<?, ?it/s]

MC x20:   0%|          | 0/4 [00:00<?, ?it/s]


 unknown


Deterministic:   0%|          | 0/2 [00:00<?, ?it/s]

MC x20:   0%|          | 0/2 [00:00<?, ?it/s]

... (27 outputs left out)


 intermediate


Deterministic:   0%|          | 0/1 [00:00<?, ?it/s]

MC x20:   0%|          | 0/1 [00:00<?, ?it/s]

{
  "seed": 2026,
  "external_test_csv_rows": 174,
  "external_test_evaluable_images": 170,
  "external_test_excluded_unmatched_label_rows": 4,
  "external_familiar_images": 98,
  "external_unknown_only_images": 50,
  "external_mixed_images": 19,
  "familiar_false_referral_rate": 0.2653061224489796,
  "normal_false_referral_rate": 0.07692307692307693,
  "known_only_false_referral_rate": 0.4782608695652174,
  "unknown_referral_sensitivity": 0.56,
  "mixed_referral_rate": 0.2631578947368421,
  "rare_to_normal_error_rate": 0.02,
  "explicit_normal_false_reassurance_rate": 0.02,
  "unknown_forced_into_known_rate": 0.42,
  "unknown_auroc": 0.7557142857142857,
  "unknown_aupr": 0.583908616885371,
  "fpr_at_95_tpr": 0.6122448979591837,
  "known_classes_with_positives": 12,
  "disease_macro_f1_all14": 0.24122900908615194,
  "disease_macro_f1_evaluable": 0.2814338439338439,
  "disease_micro_f1": 0.41739130434782606,
  "disease_mAP": 0.36125268956643786,
  "disease_macro_auroc": 0.86787467536458

In [9]:
# -----------------------------
# Aggregate results
# -----------------------------
seed_results=pd.DataFrame(summaries)
seed_results.to_csv(TABLE_DIR/"rfmid2_external_results_by_seed.csv",index=False)
display(seed_results)

metrics=[
    "familiar_false_referral_rate","normal_false_referral_rate",
    "known_only_false_referral_rate","unknown_referral_sensitivity",
    "mixed_referral_rate","rare_to_normal_error_rate",
    "explicit_normal_false_reassurance_rate","unknown_forced_into_known_rate",
    "unknown_auroc","unknown_aupr","fpr_at_95_tpr",
    "disease_macro_f1_all14","disease_macro_f1_evaluable",
    "disease_micro_f1","disease_mAP","disease_macro_auroc",
    "external_ece_before","external_ece_after",
    "external_brier_before","external_brier_after",
]

rows=[]
for m in metrics:
    v=pd.to_numeric(seed_results[m],errors="coerce")
    rows.append({
        "Metric":m,"Mean":float(v.mean()),"SD":float(v.std(ddof=1)),
        "Min":float(v.min()),"Max":float(v.max())
    })
aggregate=pd.DataFrame(rows)
aggregate.to_csv(TABLE_DIR/"rfmid2_external_mean_sd.csv",index=False)
display(aggregate)

boot=pd.DataFrame(bootstrap)
boot.to_csv(TABLE_DIR/"rfmid2_external_bootstrap_ci_by_seed.csv",index=False)
display(boot)

unknown_comp=(
    unknown_df[UNKNOWN_EXT].sum()
    .sort_values(ascending=False)
    .rename("Positive_Count").reset_index()
    .rename(columns={"index":"External_Unknown_Label"})
)
unknown_comp=unknown_comp[unknown_comp["Positive_Count"]>0]
unknown_comp.to_csv(TABLE_DIR/"rfmid2_unknown_only_label_composition.csv",index=False)
display(unknown_comp)


,seed,external_test_csv_rows,external_test_evaluable_images,external_test_excluded_unmatched_label_rows,external_familiar_images,external_unknown_only_images,external_mixed_images,familiar_false_referral_rate,normal_false_referral_rate,known_only_false_referral_rate,...,known_classes_with_positives,disease_macro_f1_all14,disease_macro_f1_evaluable,disease_micro_f1,disease_mAP,disease_macro_auroc,external_ece_before,external_ece_after,external_brier_before,external_brier_after
0,42,174,170,4,98,50,19,0.193878,0.076923,0.326087,...,12,0.241128,0.281316,0.355932,0.442477,0.861606,0.199585,0.093842,0.084160,0.068473
1,123,174,170,4,98,50,19,0.193878,0.057692,0.347826,...,12,0.240592,0.280691,0.403509,0.442876,0.934026,0.091552,0.050795,0.050421,0.048165
2,2026,174,170,4,98,50,19,0.265306,0.076923,0.478261,...,12,0.241229,0.281434,0.417391,0.361253,0.867875,0.111821,0.064007,0.058122,0.055675


,Metric,Mean,SD,Min,Max
0,familiar_false_referral_rate,0.217687,0.041239,0.193878,0.265306
1,normal_false_referral_rate,0.070513,0.011103,0.057692,0.076923
2,known_only_false_referral_rate,0.384058,0.082303,0.326087,0.478261
3,unknown_referral_sensitivity,0.513333,0.041633,0.480000,0.560000
4,mixed_referral_rate,0.263158,0.052632,0.210526,0.315789
5,rare_to_normal_error_rate,0.040000,0.020000,0.020000,0.060000
6,explicit_normal_false_reassurance_rate,0.026667,0.011547,0.020000,0.040000
7,unknown_forced_into_known_rate,0.446667,0.030551,0.420000,0.480000
8,unknown_auroc,0.759320,0.005723,0.755714,0.765918
9,unknown_aupr,0.586207,0.003707,0.583909,0.590484


,seed,Metric,Estimate,CI95_Lower,CI95_Upper
0,42,familiar_false_referral_rate,0.193878,0.122449,0.275510
1,42,unknown_referral_sensitivity,0.500000,0.360000,0.640000
2,42,rare_to_normal_error_rate,0.060000,0.000000,0.140000
3,123,familiar_false_referral_rate,0.193878,0.122449,0.275510
4,123,unknown_referral_sensitivity,0.480000,0.340000,0.620000
5,123,rare_to_normal_error_rate,0.040000,0.000000,0.100000
6,2026,familiar_false_referral_rate,0.265306,0.183673,0.357143
7,2026,unknown_referral_sensitivity,0.560000,0.420000,0.700000
8,2026,rare_to_normal_error_rate,0.020000,0.000000,0.060000


,External_Unknown_Label,Positive_Count
0,HR,13
1,EDN,12
2,RT,8
3,CWS,5
4,MS,4
5,CME,4
6,TV,3
7,TD,3
8,PRH,3
9,HTN,2


In [10]:
# -----------------------------
# Save metadata + complete ZIP
# -----------------------------

metadata = {
    "project":
        "SafeFundus",

    "external_dataset":
        "RFMiD 2.0",

    "notebook_version":
        "10B fixed mixed-case extension + exact-ID alignment",

    "primary_external_split":
        (
            "official Test_set restricted to exact "
            "CSV-ID/image-ID matches"
        ),

    "external_training_or_validation_used_for_tuning":
        False,

    "external_test_used_for_tuning":
        False,

    "classifier":
        "Swin-Tiny + ASL + explicit NORMAL",

    "open_set_method":
        (
            "Normal-Aware JointEnergy + "
            "20-pass MC Dropout"
        ),

    "fusion":
        (
            "0.5 JointEnergy + "
            "0.5 MC Dropout"
        ),

    "calibration_and_referral_threshold":
        "frozen from internal RFMiD",

    "rfmid2_mapping":
        "WNL -> NORMAL",

    "known_diseases":
        KNOWN,

    "intermediate_classes":
        INTERMEDIATE,

    "external_unknown_classes":
        UNKNOWN_EXT,

    "test_csv_rows":
        int(
            len(
                test_df
            )
        ),

    "test_image_files":
        int(
            len(
                TEST_IMAGE_INDEX
            )
        ),

    "test_evaluable_exact_matches":
        int(
            len(
                test_evaluable_df
            )
        ),

    "test_label_ids_without_exact_image":
        [
            norm_id(
                x
            )
            for x in (
                test_unmatched_labels[
                    "ID"
                ]
                .tolist()
            )
        ],

    "test_image_ids_without_label_row":
        sorted(
            list(
                set(
                    TEST_IMAGE_INDEX.keys()
                )
                -
                set(
                    test_df[
                        "_Normalized_ID"
                    ]
                    .tolist()
                )
            ),
            key=int,
        ),

    "test_extension_counts":
        TEST_EXTENSION_COUNTS,

    "image_matching_policy":
        (
            "Use actual directory inventory; extension matching "
            "is case-insensitive; require exact numeric image stem "
            "to CSV ID; unmatched pairs are reported and excluded; "
            "no manual remapping."
        ),

    "raw_test_group_counts":
        {
            k:
                int(v)
            for k, v in (
                test_df[
                    "External_Group"
                ]
                .value_counts()
                .to_dict()
                .items()
            )
        },

    "evaluable_test_group_counts":
        {
            k:
                int(v)
            for k, v in (
                test_evaluable_df[
                    "External_Group"
                ]
                .value_counts()
                .to_dict()
                .items()
            )
        },

    "csv_audit": {
        "train":
            train_csv_audit,

        "validation":
            val_csv_audit,

        "test":
            test_csv_audit,
    },
}

with open(
    OUTPUT_DIR /
    "rfmid2_external_validation_metadata.json",
    "w",
) as f:
    json.dump(
        metadata,
        f,
        indent=2,
    )


ZIP_PATH = (
    WORK_ROOT /
    f"{NOTEBOOK_STEM}.zip"
)

if ZIP_PATH.exists():
    ZIP_PATH.unlink()

shutil.make_archive(
    str(
        ZIP_PATH.with_suffix("")
    ),
    "zip",
    root_dir=OUTPUT_DIR,
)

print(
    "=" * 72
)

print(
    "COMPLETE OUTPUT ZIP CREATED"
)

print(
    "=" * 72
)

print(
    ZIP_PATH
)

print(
    f"ZIP size: "
    f"{ZIP_PATH.stat().st_size/(1024**2):.2f} MB"
)


COMPLETE OUTPUT ZIP CREATED
/kaggle/working/10B_RFMiD2_External_Validation_Fixed.zip
ZIP size: 0.21 MB


## After Notebook 10B

Upload:

**`10B_RFMiD2_External_Validation_Fixed.zip`**

This ZIP name deliberately matches this fixed notebook filename so it cannot be confused with the earlier failed Notebook 10 output.
